In [1]:
# Import Libraries
from pathlib import Path 

import pandas as pd 
import numpy as np
import lasio

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

In [2]:
# Input and output paths
ROOT      = Path().resolve().parent
RAW       = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed' 
PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
# Load well log data from LAS files and save as CSV
well_names = ["BLT-01", "EVD-01", "JUT-01", "PKP-01"]

for name in well_names:
    las = lasio.read(RAW / f"{name}.las")
    df = las.df().reset_index()
    df.to_csv(PROCESSED  / f"{name}log.csv", index=False)

In [4]:
# Load raw target lithologies data
raw_data = pd.read_csv(RAW / "target_lithologies.csv")
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3455 entries, 0 to 3454
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   well_id                3455 non-null   object 
 1   easting                3455 non-null   float64
 2   northing               3455 non-null   float64
 3   depth_tvd_m            0 non-null      float64
 4   porosity_pct           2419 non-null   float64
 5   gamma_ray_api          3455 non-null   float64
 6   bulk_density_gcc       3199 non-null   float64
 7   formation_top_tvd      3455 non-null   float64
 8   formation_base_tvd     3455 non-null   float64
 9   formation_thickness_m  3455 non-null   float64
 10  distance_to_usp_km     3455 non-null   float64
 11  flag                   3455 non-null   object 
 12  flag_reason            3455 non-null   object 
dtypes: float64(10), object(3)
memory usage: 351.0+ KB


## Checking 'well_id'

In [5]:
df = raw_data.copy() 
df['well_id'].value_counts()

well_id
BLT-01    1689
EVD-01     780
PKP-01     730
JUT-01     256
Name: count, dtype: int64

## Columns missing data

### Filling in 'depth_tvd_m'

In [6]:
# Check missing 'depth_tvd_m' values for all wells
missing_rows = df[df['depth_tvd_m'].isna()].groupby('well_id').size()
if missing_rows.empty:
    print('No missing rows') 
else:
    print("Missing depth_tvd_m counts per well:")
    print(missing_rows)

Missing depth_tvd_m counts per well:
well_id
BLT-01    1689
EVD-01     780
JUT-01     256
PKP-01     730
dtype: int64


In [7]:
def fill_depth(well_id, las_file, sheet_name, top_depth, bottom_depth, md_col, gr_col):
    # Well path data
    wellpath = pd.read_excel(RAW / 'Well Path Data.xlsx', sheet_name=sheet_name)
    wellpath_df = wellpath[['Depth (m)', 'TVD (m)']]

    # Source data from LAS file
    las = pd.read_csv(PROCESSED /las_file)

    # Subset formation interval
    subset = las[(las[md_col] >= top_depth) & (las[md_col] <= bottom_depth)]
    subset = subset[[md_col, gr_col]]

    # Interpolate TVD
    subset['depth_tvd_m'] = np.interp(subset[md_col], wellpath_df['Depth (m)'], wellpath_df['TVD (m)'])

    # Merge with target lithologies
    target_subset = df[df['well_id'] == well_id]
    df.loc[target_subset.index, 'depth_tvd_m'] = subset['depth_tvd_m'].values[:len(target_subset)]

In [8]:
# Apply tp all wells
fill_depth('BLT-01', 'BLT-01log.csv', 'BLT-01', 1924, 2052.7, 'MD', 'GR')
fill_depth('EVD-01', 'EVD-01log.csv', 'EVD-01', 1788, 1866, 'DEPT', 'GR')
fill_depth('JUT-01', 'JUT-01log.csv', 'JUT-01', 1659.5, 1787, 'DEPT:1', 'GR')
fill_depth('PKP-01', 'PKP-01log.csv', 'PKP-01', 2530.5, 2603.5, 'DEPT', 'GR')

In [9]:
# Check missing 'depth_tvd_m' values for all wells
missing_rows = df[df['depth_tvd_m'].isna()].groupby('well_id').size()
if missing_rows.empty:
    print('No missing rows') 
else:
    print("Missing depth_tvd_m counts per well:")
    print(missing_rows)

No missing rows


### Filling in 'bulk_density_gcc'

In [10]:
# Check missing 'bulk_density_gcc' values for all wells
missing_rows = df[df['bulk_density_gcc'].isna()].groupby('well_id').size()
if missing_rows.empty:
    print('No missing rows') 
else:
    print("Missing bulk_density_gcc counts per well:")
    print(missing_rows)

Missing bulk_density_gcc counts per well:
well_id
JUT-01    256
dtype: int64


In [11]:
# Well 'JUT-01'
target_well = 'JUT-01'

# Selected features
features = [
    'gamma_ray_api',
    'depth_tvd_m',
    'formation_thickness_m',
]

df['formation_thickness_m'] = (
    df['formation_base_tvd'] - df['formation_top_tvd']
)

# Train subset consisting of 'EVD-01', 'BLT-01', 'PKP-01'
train_df = df[
    (df["well_id"] != target_well) &
    (df["bulk_density_gcc"].notna())
].copy()

train_df = train_df.dropna(subset=features)

X      = train_df[features]
y      = train_df['bulk_density_gcc']

# Train test split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
)
rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)
mae = mean_absolute_error(y_val, y_pred_val)
print(f"\nValidation MAE : {mae:.4f} g/cc")

target_mask = (
    (df['well_id'] == target_well) & 
    (df['bulk_density_gcc'].isna()))
X_target    = df.loc[target_mask, features].copy()

# Rows where all features are present
predictable_idx  = X_target.dropna().index
unpredictable_n  = len(X_target) - len(predictable_idx)

if unpredictable_n > 0:
    print(f"\n⚠ Warning: {unpredictable_n} target rows skipped (missing feature values).")

if len(predictable_idx) == 0:
    print("⚠ No predictable rows found for the target well. Check feature availability.")
else:
    df.loc[predictable_idx, "bulk_density_gcc"] = rf.predict(X_target.loc[predictable_idx])
    print(f"\n✓ Predicted bulk density for {len(predictable_idx)} rows in {target_well}.")


Validation MAE : 0.0065 g/cc

✓ Predicted bulk density for 256 rows in JUT-01.


In [12]:
# Check missing 'bulk_density_gcc' values for all wells
missing_rows = df[df['bulk_density_gcc'].isna()].groupby('well_id').size()
if missing_rows.empty:
    print('No missing rows') 
else:
    print("Missing bulk_density_gcc counts per well:")
    print(missing_rows)

No missing rows


### Filling in 'porosity_pct'

In [13]:
# Check missing 'porosity_pct' values for all wells
missing_rows = df[df['porosity_pct'].isna()].groupby('well_id').size()
if missing_rows.empty:
    print('No missing rows') 
else:
    print("Missing porosity_pct counts per well:")
    print(missing_rows)

Missing porosity_pct counts per well:
well_id
EVD-01    780
JUT-01    256
dtype: int64


$$\text{Porosity - Density Equation}$$
$$\phi = \frac{\rho_{ma} - \rho_b}{\rho_{ma} - \rho_{fl}} \times 100$$

In [14]:
# Well 'EVD-01'
# Slochteren Sandstone constants
rho_ma = 2.65  # g/cc -quartz sandstone matrix density(2.65, 2.68)
rho_fl = 1.07  # g/cc -saline formation brine density(1.07, 1.1)

target_subset_evd = df[df['well_id'] == 'EVD-01']
df.loc[target_subset_evd.index, 'porosity_pct'] =(
    ((rho_ma - df.loc[target_subset_evd.index, 'bulk_density_gcc']) / (rho_ma - rho_fl))
    .clip(0.02, 1) 
    * 100
    )

In [15]:
# Check missing 'porosity_pct' values for all wells
missing_rows = df[df['porosity_pct'].isna()].groupby('well_id').size()
if missing_rows.empty:
    print('No missing rows') 
else:
    print("Missing porosity_pct counts per well:")
    print(missing_rows)

Missing porosity_pct counts per well:
well_id
JUT-01    256
dtype: int64


In [16]:
# Well 'JUT-01'
target_well = 'JUT-01'

# Train subset consisting of 'EVD-01', 'BLT-01', 'PKP-01'
train_df = df[
    (df["well_id"] != target_well) &
    (df["porosity_pct"].notna())
].copy()

train_df = train_df.dropna(subset=features)

X      = train_df[features]
y      = train_df['porosity_pct']

# Train test split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Random forest
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
)
rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)
mae = mean_absolute_error(y_val, y_pred_val)
print(f"\nValidation MAE : {mae:.4f} %")

target_mask = (
    (df["well_id"] == target_well) & 
    (df["porosity_pct"].isna()))
X_target    = df.loc[target_mask, features].copy()

# Rows where all features are present
predictable_idx  = X_target.dropna().index
unpredictable_n  = len(X_target) - len(predictable_idx)

if unpredictable_n > 0:
    print(f"\n⚠ Warning: {unpredictable_n} target rows skipped (missing feature values).")

if len(predictable_idx) == 0:
    print("⚠ No predictable rows found for the target well. Check feature availability.")
else:
    df.loc[predictable_idx, "porosity_pct"] = rf.predict(X_target.loc[predictable_idx])
    print(f"\n✓ Predicted porosity for {len(predictable_idx)} rows in {target_well}.")


Validation MAE : 0.3134 %

✓ Predicted porosity for 256 rows in JUT-01.


In [17]:
# Check missing 'porosity_pct' values for all wells
missing_rows = df[df['porosity_pct'].isna()].groupby('well_id').size()
if missing_rows.empty:
    print('No missing rows') 
else:
    print("Missing porosity_pct counts per well:")
    print(missing_rows)

No missing rows


In [18]:
df_filled = df.copy()
df_filled.info()
df_filled.to_csv(PROCESSED /'target_lithologies_filled.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3455 entries, 0 to 3454
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   well_id                3455 non-null   object 
 1   easting                3455 non-null   float64
 2   northing               3455 non-null   float64
 3   depth_tvd_m            3455 non-null   float64
 4   porosity_pct           3455 non-null   float64
 5   gamma_ray_api          3455 non-null   float64
 6   bulk_density_gcc       3455 non-null   float64
 7   formation_top_tvd      3455 non-null   float64
 8   formation_base_tvd     3455 non-null   float64
 9   formation_thickness_m  3455 non-null   float64
 10  distance_to_usp_km     3455 non-null   float64
 11  flag                   3455 non-null   object 
 12  flag_reason            3455 non-null   object 
dtypes: float64(10), object(3)
memory usage: 351.0+ KB
